<a href="https://colab.research.google.com/github/treborskrub/Multi-agent-/blob/main/biepv4agnt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

#!/usr/bin/env python3
# -*- coding: utf-8 -*-

from __future__ import annotations
from dataclasses import dataclass, asdict, field
from enum import Enum
from typing import Any, Dict, List, Callable


# =========================
# ENUMS & DATA STRUCTURES
# =========================

class Stance(Enum):
    DEEPEN = "DEEPEN"
    EXPLORE = "EXPLORE"
    STABILIZE = "STABILIZE"

    def to_json(self):
        return self.value


@dataclass
class DriveLevels:
    S: float = 0.0
    T: float = 0.0
    M: float = 0.0

    def __add__(self, other: "DriveLevels") -> "DriveLevels":
        return DriveLevels(self.S + other.S, self.T + other.T, self.M + other.M)

    def normalize(self) -> "DriveLevels":
        total = self.S + self.T + self.M
        if total == 0:
            return self
        return DriveLevels(self.S / total, self.T / total, self.M / total)

    def to_json(self):
        return asdict(self)


@dataclass
class InterpretiveMetrics:
    IG1: float = 0.0
    II1: float = 0.0

    def to_json(self):
        return asdict(self)


@dataclass
class PassResult:
    name: str
    description: str
    drives: DriveLevels
    stance_suggestion: Stance
    guide: Stance
    majority_stance: Stance
    notes: Dict[str, Any] = field(default_factory=dict)

    def to_json(self):
        return {
            "name": self.name,
            "description": self.description,
            "drives": self.drives.to_json(),
            "stance_suggestion": self.stance_suggestion.to_json(),
            "guide": self.guide.to_json(),
            "majority_stance": self.majority_stance.to_json(),
            "notes": self.notes,
        }


@dataclass
class BEIPReport:
    object_label: str
    passes: List[PassResult]
    metrics: InterpretiveMetrics
    final_stance: Stance
    meta_summary: str

    def to_dict(self) -> Dict[str, Any]:
        return {
            "object_label": self.object_label,
            "passes": [p.to_json() for p in self.passes],
            "metrics": self.metrics.to_json(),
            "final_stance": self.final_stance.to_json(),
            "meta_summary": self.meta_summary,
        }


# =========================
# TASK STATE & TOOLS
# =========================

@dataclass
class TaskState:
    label: str
    context: Dict[str, Any] = field(default_factory=dict)
    data: Any = None
    log: List[Dict[str, Any]] = field(default_factory=list)

    def record(self, event: str, **fields: Any):
        entry = {"event": event}
        entry.update(fields)
        self.log.append(entry)


Tool = Callable[[TaskState, Dict[str, Any]], Any]


class ToolLayer:
    def __init__(self):
        self.tools: Dict[str, Tool] = {}

    def register(self, name: str, fn: Tool):
        self.tools[name] = fn

    def call(self, name: str, state: TaskState, **kwargs):
        tool = self.tools.get(name)
        if tool is None:
            state.record("tool_missing", tool=name)
            return None
        state.record("tool_call", tool=name, kwargs=kwargs)
        return tool(state, kwargs)


# =========================
# SPECIALIST AGENTS
# =========================

class StructuralAgent:
    def run(self, state: TaskState, tools: ToolLayer):
        state.record("structural_agent_start")
        tools.call("analyze_structure", state)
        state.record("structural_agent_end")


class ExplorerAgent:
    def run(self, state: TaskState, tools: ToolLayer):
        state.record("explorer_agent_start")
        tools.call("search", state, query=state.context.get("query", ""))
        state.record("explorer_agent_end")


class SafetyVerifierAgent:
    def run(self, state: TaskState, tools: ToolLayer):
        state.record("safety_agent_start")
        tools.call("verify", state)
        state.record("safety_agent_end")


# =========================
# BEIP SUPERVISOR (v4)
# =========================

class BEIPSupervisor:
    def __init__(self):
        self.metrics = InterpretiveMetrics()
        self.structural_agent = StructuralAgent()
        self.explorer_agent = ExplorerAgent()
        self.safety_agent = SafetyVerifierAgent()

    def _triadic_majority(self, d: DriveLevels) -> Stance:
        dominant = max({"S": d.S, "T": d.T, "M": d.M}, key=lambda k: {"S": d.S, "T": d.T, "M": d.M}[k])
        return {
            "S": Stance.DEEPEN,
            "T": Stance.EXPLORE,
            "M": Stance.STABILIZE,
        }[dominant]

    def _update_metrics(self, ig: float, ii: float):
        self.metrics.IG1 += ig
        self.metrics.II1 += ii

    # ---- PASSES ----

    def _structural_pass(self, state: TaskState) -> PassResult:
        d = DriveLevels(1.0, 0.3, 0.4)
        self._update_metrics(0.2, 0.1)
        return PassResult(
            "Structural",
            "Identify core components, topology, and invariants.",
            d,
            Stance.DEEPEN,
            Stance.DEEPEN,
            self._triadic_majority(d),
        )

    def _temporal_pass(self, state: TaskState) -> PassResult:
        d = DriveLevels(0.5, 1.0, 0.6)
        self._update_metrics(0.4, 0.1)
        return PassResult(
            "Temporal",
            "Analyze evolution, dynamics, and trajectories.",
            d,
            Stance.EXPLORE,
            Stance.EXPLORE,
            self._triadic_majority(d),
        )

    def _behavioral_pass(self, state: TaskState) -> PassResult:
        d = DriveLevels(0.4, 0.9, 1.0)
        guide = Stance.STABILIZE if self.metrics.IG1 > 0.5 else Stance.EXPLORE
        self._update_metrics(0.3, 0.3)
        return PassResult(
            "Behavioral",
            "Characterize emergent patterns: drift, resonance, collapse.",
            d,
            guide,
            guide,
            self._triadic_majority(d),
        )

    def _meta_pass(self, state: TaskState) -> PassResult:
        d = DriveLevels(0.6, 0.6, 1.2)
        guide = Stance.STABILIZE if self.metrics.II1 > 0.5 else Stance.DEEPEN
        self._update_metrics(-0.2, 0.4)
        return PassResult(
            "Meta",
            "Synthesize identity, role, and global meaning.",
            d,
            guide,
            guide,
            self._triadic_majority(d),
        )

    # ---- AUDIT ----

    def audit(self, state: TaskState) -> BEIPReport:
        self.metrics = InterpretiveMetrics()

        passes = [
            self._structural_pass(state),
            self._temporal_pass(state),
            self._behavioral_pass(state),
            self._meta_pass(state),
        ]

        total = DriveLevels()
        for p in passes:
            total += p.drives
        total = total.normalize()

        stance = self._triadic_majority(total)

        if self.metrics.IG1 > 0.8:
            stance = Stance.STABILIZE

        summary = (
            f"BEIP audit complete. Final stance: {stance.value}. "
            f"Drives: {total.to_json()}. Metrics: {self.metrics.to_json()}."
        )

        return BEIPReport(
            object_label=state.label,
            passes=passes,
            metrics=self.metrics,
            final_stance=stance,
            meta_summary=summary,
        )

    def route(self, report: BEIPReport, state: TaskState, tools: ToolLayer):
        stance = report.final_stance
        state.record("routing", stance=stance.value)

        if stance == Stance.DEEPEN:
            self.structural_agent.run(state, tools)
        elif stance == Stance.EXPLORE:
            self.explorer_agent.run(state, tools)
        else:
            self.safety_agent.run(state, tools)


# =========================
# EXAMPLE TOOLS & EXECUTION
# =========================

def tool_search(state, params):
    state.record("search_tool", query=params.get("query"))
    return {"results": ["example"]}

def tool_analyze(state, params):
    state.record("analyze_tool")
    return {"structure": "ok"}

def tool_verify(state, params):
    state.record("verify_tool")
    return {"verified": True}


if __name__ == "__main__":
    import json

    task = TaskState("Example Task", context={"query": "Navier-Stokes"})
    tools = ToolLayer()
    tools.register("search", tool_search)
    tools.register("analyze_structure", tool_analyze)
    tools.register("verify", tool_verify)

    supervisor = BEIPSupervisor()
    report = supervisor.audit(task)
    supervisor.route(report, task, tools)

    print(json.dumps(report.to_dict(), indent=2))
    print(json.dumps(task.log, indent=2))

{
  "object_label": "Example Task",
  "passes": [
    {
      "name": "Structural",
      "description": "Identify core components, topology, and invariants.",
      "drives": {
        "S": 1.0,
        "T": 0.3,
        "M": 0.4
      },
      "stance_suggestion": "DEEPEN",
      "guide": "DEEPEN",
      "majority_stance": "DEEPEN",
      "notes": {}
    },
    {
      "name": "Temporal",
      "description": "Analyze evolution, dynamics, and trajectories.",
      "drives": {
        "S": 0.5,
        "T": 1.0,
        "M": 0.6
      },
      "stance_suggestion": "EXPLORE",
      "guide": "EXPLORE",
      "majority_stance": "EXPLORE",
      "notes": {}
    },
    {
      "name": "Behavioral",
      "description": "Characterize emergent patterns: drift, resonance, collapse.",
      "drives": {
        "S": 0.4,
        "T": 0.9,
        "M": 1.0
      },
      "stance_suggestion": "STABILIZE",
      "guide": "STABILIZE",
      "majority_stance": "STABILIZE",
      "notes": {}
    },
 